<a href="https://colab.research.google.com/github/ryanaxiom/Applied-ML/blob/main/code/Day10_Carnegie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Student Instructions**: Before you begin, click **File > Save a copy in Drive** so you do not lose your work!

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold
from sklearn.model_selection import cross_val_score, cross_val_predict

url = ("https://raw.githubusercontent.com/ryanaxiom/Applied-ML/main/data/carnegie_data.csv")

carnegie_raw = pd.read_csv(url, encoding="cp1252")
carnegie = carnegie_raw.copy()
carnegie["research"] = carnegie["serd"] + carnegie["nonserd"]
d = carnegie[carnegie["research"].notna()].copy()
d["med_yn"] = (d["medical"] == 1).astype(int)
y = d["research"]

two  = ["fallenr20", "med_yn"]
trio = ["fallenr20", "med_yn", "stem_rsd"]
pile = ["fallenr20", "totdeg", "facnum", "stem_rsd", "med_yn"]
def rmse(a, b): return np.sqrt(np.mean((a - b)**2))

In [ ]:
tests, tested = [], pd.Series(0, index=d.index)
for seed in range(5):
    tr, te = train_test_split(d, test_size=0.25, random_state=seed)
    m = LinearRegression()
    m.fit(tr[pile], tr["research"])
    tests.append(rmse(te["research"], m.predict(te[pile])))
    tested[te.index] += 1            # who landed in a test set?
np.mean(tests), (tested == 0).sum(), (tested >= 3).sum()

(np.float64(196289.82519645483), np.int64(76), np.int64(39))

In [ ]:
# how often was Johns Hopkins tested?
tested[d["name"] == "Johns Hopkins University"]

,0
1582,2


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=309)
folds = []
for train_idx, test_idx in kf.split(d):
    tr, te = d.iloc[train_idx], d.iloc[test_idx]
    m = LinearRegression()
    m.fit(tr[pile], tr["research"])
    folds.append(rmse(te["research"], m.predict(te[pile])))
np.round(folds), np.mean(folds)

(array([206944., 142656., 143561., 299353., 175758.]),
 np.float64(193654.3696470377))

Plot comparison of random selection of train/test directly versus KFold cv with equal proportions of train/test

In [ ]:
# which fold got Johns Hopkins?
for k, (train_idx, test_idx) in enumerate(kf.split(d), start=1):
    if "Johns Hopkins University" in d.iloc[test_idx]["name"].values:
        print("fold", k)

fold 4


In [ ]:
model = LinearRegression()
scores = cross_val_score(model, d[pile], y, cv=kf,
                         scoring="neg_root_mean_squared_error")
np.round(scores)

array([-206944., -142656., -143561., -299353., -175758.])

In [ ]:
model = LinearRegression()
oof = cross_val_predict(model,d[pile],y,cv=kf)
rmse(y,oof)

np.float64(201981.36833312988)

In [ ]:
np.allclose(-scores, folds)     # the shortcut matches the by-hand loop

True

In [ ]:
-scores==folds

array([ True,  True,  True,  True,  True])

In [ ]:
# the three numbers side by side
model.fit(d[pile], y)
print("pooled CV RMSE   ", round(rmse(y, oof)))
print("mean of fold RMSE", round(np.mean(folds)))
print("in-sample RMSE   ", round(rmse(y, model.predict(d[pile]))))

pooled CV RMSE    201981
mean of fold RMSE 193654
in-sample RMSE    194051


In [ ]:
d["oof_resid"] = y - oof
who = ["Massachusetts Institute of Technology", "Marshall University",
       "University of Pittsburgh-Pittsburgh Campus", "Johns Hopkins University"]
d[d["name"].isin(who)][["name", "research", "oof_resid"]]

,name,research,oof_resid
1582,Johns Hopkins University,3110494.0,2.053414e+06
1859,Marshall University,22250.0,-1.103300e+05
1877,Massachusetts Institute of Technology,987968.0,8.749731e+04
3541,University of Pittsburgh-Pittsburgh Campus,1105532.0,-3.681389e+04


In [ ]:
rng = np.random.default_rng(309)
for j in range(10):
    d[f"junk{j}"] = rng.normal(size=len(d))
junk = pile + [f"junk{j}" for j in range(10)]
for name, cols in [("two", two), ("trio", trio), ("pile", pile), ("pile+junk", junk)]:
    model = LinearRegression()
    model.fit(d[cols], y)
    print(name, round(rmse(y, model.predict(d[cols]))),
                round(rmse(y, cross_val_predict(model, d[cols], y, cv=kf))))

two 314590 316171
trio 233132 236453
pile 194051 201981
pile+junk 191289 211109


In [ ]:
for K in [5, 10, 302]:
    kk = KFold(n_splits=K, shuffle=True, random_state=309)
    print(K, round(rmse(y, cross_val_predict(LinearRegression(), d[pile], y, cv=kk))))

5 201981
10 200871
302 201112


In [ ]:
# and why the mean-of-folds version cannot be trusted: at K = n it is MAE
for K in [5, 10, 302]:
    kk = KFold(n_splits=K, shuffle=True, random_state=309)
    s = cross_val_score(LinearRegression(), d[pile], y, cv=kk, scoring="neg_root_mean_squared_error")
    print("mean of ",K," folds:", round(-s.mean()))
print("MAE of the leave-one-out predictions:", round(np.mean(np.abs(y - cross_val_predict(LinearRegression(), d[pile], y, cv=KFold(302, shuffle=True, random_state=309))))))

mean of  5  folds: 193654
mean of  10  folds: 183496
mean of  302  folds: 106215
MAE of the leave-one-out predictions: 106215


Add a plot showing the convergence to MAE of leave one out.
Also empirically (or theoretically) show the distribution of estimates on the same scale for 5, 10, 302.

In [ ]:
single, cv5 = [], []
for s in range(200):
    tr, te = train_test_split(d, test_size=0.25, random_state=s)
    m = LinearRegression()
    m.fit(tr[pile], tr["research"])
    single.append(rmse(te["research"], m.predict(te[pile])))
    kk = KFold(n_splits=5, shuffle=True, random_state=s)
    cv5.append(rmse(y, cross_val_predict(LinearRegression(), d[pile], y, cv=kk)))
np.ptp(single), np.ptp(cv5)   # range = max - min

(np.float64(222308.18505844055), np.float64(13157.728459599923))